In [ ]:
# hide
import numpy as np
import pyquist as pq

In [ ]:
# hide
def variable_feedback_delay(x: np.ndarray, M: np.ndarray, g: float = 1.0) -> np.ndarray:
    """A feedback delay line with a time-varying, fractional delay M[n] >= 1:
    y[n] = x[n] + g * y[n - M[n]]."""
    L = int(np.ceil(M.max())) + 2     # a delay line long enough for the longest delay
    buffer = np.zeros(L)
    i = 0                             # where y[n] will be written
    y = np.zeros(len(x))
    for n in range(len(x)):
        k = int(M[n])                 # whole samples of delay...
        a = M[n] - k                  # ...and the fraction left over
        # interpolate between y[n - k] and y[n - k - 1], like a wavetable (Chapter 3)
        delayed = (1 - a) * buffer[(i - k) % L] + a * buffer[(i - k - 1) % L]
        y[n] = x[n] + g * delayed
        buffer[i] = y[n]
        i = (i + 1) % L
    return y


def wet_ratio(db: float) -> float:
    """The wet ratio w that mixes the wet signal `db` decibels above the dry signal."""
    return 1 / (1 + pq.helper.db_to_amplitude(-db))

In [ ]:
def flanger(x: np.ndarray, M_c: float, A: float, f_m: float, g: float,
            f_s: int) -> np.ndarray:
    """A feedback delay line whose delay sweeps around M_c samples:
    M[n] = M_c (1 + A sin(2 pi f_m n / f_s))."""
    n = np.arange(len(x))
    M = M_c * (1 + A * np.sin(2 * np.pi * f_m * n / f_s))
    return variable_feedback_delay(x, M, g)


pad = pq.Audio.from_file("../assets/pad.wav")
guitar = pq.Audio.from_file("../assets/guitar-distorted.wav")
w = wet_ratio(0.0)                            # equal parts wet and dry
for sound in [pad, guitar]:
    x = sound.samples[:, 0]
    f_s = sound.sample_rate
    wet = flanger(x, M_c=0.003 * f_s, A=0.9, f_m=0.25, g=0.7, f_s=f_s)   # around 3 ms, every 4 s
    pq.play(sound, normalize=True)
    pq.play(pq.Audio((1 - w) * x + w * wet, f_s), normalize=True)